# Fish Audio S2 Pro — دبلجة عربية احترافية

هذا الدفتر يشغّل مسار Fish Audio S2 Pro من مستودع `dub22` على Google Colab. شغّل الخلايا بالترتيب. النموذج يحتاج GPU قويًا؛ إذا لم تتوفر GPU مناسبة سيتوقف الدفتر برسالة واضحة بدل ظهور خطأ غامض.

> لا تضع مفاتيح أو رموزًا سرية داخل الخلايا أو المستودع. الأوزان تُنزّل إلى جلسة Colab المؤقتة فقط، ويمكن تنزيل الفيديو الناتج إلى جهازك.

## 1) تفعيل GPU والتحقق من الموارد

من قائمة Colab اختر: **Runtime → Change runtime type → T4 GPU أو GPU أقوى**. يفضّل GPU بذاكرة 24GB أو أكثر لتشغيل S2 Pro.

In [ ]:
import os, re, subprocess, sys

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'], capture_output=True, text=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError('لم يتم العثور على GPU. من إعدادات Colab اختر Runtime > Change runtime type > GPU ثم أعد تشغيل الخلية.')
print(gpu.stdout.strip())
m = re.search(r'([0-9]+) MiB', gpu.stdout)
if m and int(m.group(1)) < 20000:
    raise RuntimeError('ذاكرة GPU أقل من المطلوب عمليًا لـ Fish S2 Pro. استخدم GPU بذاكرة تقارب 24GB أو أكثر.')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('PyTorch لا يرى CUDA. أعد تشغيل جلسة Colab بعد اختيار GPU.')
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))


## 2) تنزيل workspace وتثبيت بيئة Fish S2 Pro

هذه الخلية تستخدم نسخة Fish الرسمية وتثبت الاعتمادات داخل بيئة معزولة. إذا أعدت تشغيل الدفتر فلن تعيد التنزيل ما دامت الجلسة نفسها ما زالت حية.

In [ ]:
%cd /content
!apt-get update -qq
!apt-get install -y -qq ffmpeg libsox-dev portaudio19-dev
!pip -q install uv
!rm -rf /content/dub22
!git clone --depth 1 https://github.com/dhiyaddineb-hue/dub22.git /content/dub22
!mkdir -p /content/dub22/vendor
!git clone --depth 1 https://github.com/fishaudio/fish-speech.git /content/dub22/vendor/fish-speech
%cd /content/dub22/vendor/fish-speech
!uv sync --extra cu126 --no-dev
!export PATH="/content/dub22/vendor/fish-speech/.venv/bin:$PATH"
%cd /content/dub22
print('تم تجهيز workspace وبيئة Fish S2 Pro.')


## 3) تنزيل أوزان Fish S2 Pro

الأوزان كبيرة، ولذلك قد تستغرق عدة دقائق. التنزيل العام لا يحتاج مفتاح Hugging Face، لكن وضع `HF_TOKEN` اختياري لتجاوز حدود الطلبات المجهولة.

In [ ]:
%cd /content/dub22/vendor/fish-speech
!mkdir -p checkpoints/s2-pro
!vendor/fish-speech/.venv/bin/hf download fishaudio/s2-pro --local-dir checkpoints/s2-pro
import os
required = ['codec.pth']
missing = [x for x in required if not os.path.exists('/content/dub22/vendor/fish-speech/checkpoints/s2-pro/' + x)]
if missing:
    raise FileNotFoundError('لم يكتمل تنزيل أوزان Fish S2 Pro: ' + ', '.join(missing))
print('تم تنزيل أوزان Fish S2 Pro.')
%cd /content/dub22


## 4) اختيار الفيديو

يمكنك استخدام فيديو الاختبار الموجود في المستودع، أو رفع فيديو جديد. ارفع ملف MP4 واحدًا فقط لتجنب اختيار ملف خاطئ.

In [ ]:
from pathlib import Path
from google.colab import files

default_video = Path('/content/dub22/assets/input/new_job/source.mp4')
use_upload = False  # غيّرها إلى True لرفع فيديو آخر
if use_upload:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError('ارفع ملف فيديو واحدًا فقط.')
    uploaded_name = next(iter(uploaded))
    input_video = Path('/content/dub22/assets/input/colab_job') / uploaded_name
    input_video.parent.mkdir(parents=True, exist_ok=True)
    input_video.write_bytes(uploaded[uploaded_name])
else:
    input_video = default_video

if not input_video.exists():
    raise FileNotFoundError(f'الفيديو غير موجود: {input_video}')
print('الفيديو المختار:', input_video)


## 5) التحقق من الـ manifest والمراجع الصوتية

يحتوي الـ manifest على النص العربي، التوقيتات، النص الإنجليزي المرجعي، ومقاطع الصوت الخاصة بكل متحدث. لا تبدأ التوليد إذا كان أي ملف مفقودًا.

In [ ]:
import json
manifest_path = Path('/content/dub22/manifests/new_job/dialogue_ar_fish_s2.json')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
if not manifest.get('segments'):
    raise ValueError('الـ manifest لا يحتوي على مقاطع حوار.')
for segment in manifest['segments']:
    ref = Path('/content/dub22') / segment['reference_audio']
    if not ref.exists():
        raise FileNotFoundError(f'مرجع الصوت مفقود: {ref}')
    if segment['source_end'] <= segment['start']:
        raise ValueError(f'توقيت غير صالح في {segment["id"]}')
print(f'تم التحقق من {len(manifest["segments"])} مقطعًا صوتيًا ومراجعها.')


## 6) تشغيل Fish S2 Pro

سيستخدم هذا الأمر GPU. قد يستغرق التوليد وقتًا بحسب نوع GPU وطول الفيديو. لا تغلق جلسة Colab أثناء التشغيل.

In [ ]:
%cd /content/dub22
output_video = Path('/content/dub22/outputs/new_job/arabic_dub_fish_s2_pro.mp4')
output_video.parent.mkdir(parents=True, exist_ok=True)
cmd = [
    '/content/dub22/vendor/fish-speech/.venv/bin/python', 'scripts/fish_s2_dub.py',
    '--input', str(input_video),
    '--manifest', str(manifest_path),
    '--output', str(output_video),
    '--fish-root', 'vendor/fish-speech',
    '--python', 'vendor/fish-speech/.venv/bin/python',
    '--checkpoint', 'checkpoints/s2-pro',
    '--workdir', 'assets/fish_s2/colab_job',
    '--device', 'cuda',
    '--temperature', '0.75',
    '--top-p', '0.85',
    '--max-new-tokens', '1400',
]
print('بدء التوليد...')
result = subprocess.run(cmd, cwd='/content/dub22', text=True)
if result.returncode != 0:
    raise RuntimeError('فشل Fish S2 Pro. راجع آخر رسائل الخلية، وتأكد من GPU والأوزان.')
if not output_video.exists() or output_video.stat().st_size == 0:
    raise RuntimeError('لم يُنتج Fish S2 Pro ملف فيديو.')
print('اكتمل التوليد:', output_video)


## 7) فحص الفيديو وتنزيله

تتحقق الخلية من أن الملف يحتوي على مساري فيديو وصوت وأنه قابل للقراءة قبل تنزيله.

In [ ]:
probe = subprocess.run([
    'ffprobe', '-v', 'error', '-show_entries', 'format=duration:stream=codec_type,codec_name',
    '-of', 'default=noprint_wrappers=1', str(output_video)
], capture_output=True, text=True, check=True)
print(probe.stdout)
subprocess.run(['ffmpeg', '-v', 'error', '-i', str(output_video), '-f', 'null', '-'], check=True)
from IPython.display import Video, display
display(Video(str(output_video), embed=False))
files.download(str(output_video))
